# Analisis Decision Tree

## Tahap 0 - Import library

In [ ]:
from IPython.display import Image, display
import pandas as pd
import train_decision_tree as dt

## Tahap 1 - Membaca dataset

In [ ]:
df, X, y, groups, feature_columns = dt.load_dataset()
labels = sorted(y.unique())

print('Jumlah data:', len(df))
print('Jumlah baterai:', df[dt.GROUP_COLUMN].nunique())
print('Jumlah fitur:', len(feature_columns))
print('Kelas target:', labels)
display(df.head())

## Tahap 2 - EDA

In [ ]:
dt.make_eda_plots(df)

for path in [dt.EDA_CLASS_PATH, dt.EDA_SOH_PATH, dt.EDA_CORR_PATH]:
    print(path)
    display(Image(filename=str(path)))

print(dt.EDA_NOTE_PATH.read_text(encoding='utf-8'))

## Tahap 3 - Preprocessing

In [ ]:
dt.save_preprocessing_outputs(df, feature_columns)

print(dt.DATASET_SUMMARY_PATH.read_text(encoding='utf-8'))
display(pd.read_csv(dt.MISSING_VALUES_PATH))
display(pd.read_csv(dt.SELECTED_FEATURES_PATH))
display(pd.read_csv(dt.EXCLUDED_FEATURES_PATH))

## Tahap 4 - Pipeline dan parameter

In [ ]:
param_grid = {
    'model__criterion': ['gini', 'entropy'],
    'model__max_depth': [3, 4, 5, 6, 7, 8],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 3, 5, 10],
}

dt.build_pipeline()

## Tahap 5 - Validasi model

In [ ]:
oof_pred, fold_results = dt.run_nested_group_cv(X, y, groups, labels, param_grid)
display(fold_results)
display(fold_results[['test_accuracy', 'test_f1_weighted', 'test_precision_weighted', 'test_recall_weighted', 'overfit_gap_f1']].mean())

## Tahap 6 - Melatih model akhir

In [ ]:
final_grid = dt.train_final_model(X, y, groups, param_grid)
print('Parameter terbaik:', final_grid.best_params_)

## Tahap 7 - Evaluasi

In [ ]:
dt.save_model_outputs(
    final_model=final_grid.best_estimator_,
    y=y,
    oof_pred=oof_pred,
    labels=labels,
    final_grid=final_grid,
    fold_results=fold_results,
    df=df,
    feature_columns=feature_columns,
)

print(dt.METRICS_PATH.read_text(encoding='utf-8'))

## Tahap 8 - Gambar hasil

In [ ]:
for path in [dt.CONFUSION_MATRIX_PATH, dt.FEATURE_IMPORTANCE_PATH, dt.TREE_PATH]:
    print(path)
    display(Image(filename=str(path)))

## Tahap 9 - Interpretasi model

In [ ]:
display(pd.read_csv(dt.FEATURE_IMPORTANCE_CSV_PATH))
print(dt.RULES_PATH.read_text(encoding='utf-8'))